# Day 27：Memory 记忆机制（滑动窗口、摘要压缩与 SQLite 会话隔离）

> **版本**：`v0.3.0-day27` · **关联需求**：Issue #027（多会话记忆隔离）

本 Notebook 演示如何从 `project_iterations/day_27` 导入智枢记忆引擎，依次验证：
1. `InMemoryChatMessageHistory` 与 `WindowMemory(k=2)` 滑动窗口裁剪
2. `SummaryMemory` 早期对话摘要压缩
3. `SQLiteChatMessageHistory` 持久化落盘与多租户 `session_id` 隔离
4. `RunnableWithMessageHistory` 端到端多轮对话闭环

## 1. 环境初始化与导入 `project_iterations/day_27` 模块
使用 `tempfile.TemporaryDirectory` 创建隔离的临时 SQLite 数据库路径，避免污染仓库。

In [1]:
import sys
import tempfile
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day27_dir = repo_root / "project_iterations" / "day_27"
if str(day27_dir) not in sys.path:
    sys.path.insert(0, str(day27_dir))

from zhishu_agent import (
    InMemoryChatMessageHistory,
    RunnableWithMessageHistory,
    SQLiteChatMessageHistory,
    SummaryMemory,
    WindowMemory,
)

tmp_dir = tempfile.TemporaryDirectory()
db_path = Path(tmp_dir.name) / "day27_notebook.db"
print("SQLite 临时路径:", db_path)

SQLite 临时路径: /tmp/tmp3g36tqst/day27_notebook.db


## 2. 验证 `WindowMemory` 滑动窗口裁剪与 System Prompt 保护
写入 1 条 `system` 消息与 4 轮对话（8 条），设置 `WindowMemory(k=2, keep_system=True)`，观察保留结果。

In [2]:
mem_history = InMemoryChatMessageHistory()
mem_history.add_system_message("你是智枢企业知识库助手，回答需严谨简洁。")
for i in range(1, 5):
    mem_history.add_user_message(f"第 {i} 轮提问")
    mem_history.add_ai_message(f"第 {i} 轮回答")

window = WindowMemory(k=2, keep_system=True)
trimmed = window.load_memory_variables(mem_history)
print(f"原始消息数: {len(mem_history.messages)} -> 裁剪后消息数: {len(trimmed)}")
for msg in trimmed:
    print(f"  [{msg.role}] {msg.content}")

原始消息数: 9 -> 裁剪后消息数: 5
  [system] 你是智枢企业知识库助手，回答需严谨简洁。
  [user] 第 3 轮提问
  [assistant] 第 3 轮回答
  [user] 第 4 轮提问
  [assistant] 第 4 轮回答


## 3. 验证 `SummaryMemory` 早期历史摘要压缩
保留最近 2 条消息，将更早的对话自动压缩为一条 `system` 摘要消息。

In [3]:
summary_mem = SummaryMemory(max_recent_messages=2)
summary_vars = summary_mem.load_memory_variables(mem_history)
print("生成摘要:", summary_vars["summary"])
print("合并后上下文条数:", len(summary_vars["messages"]))
for msg in summary_vars["messages"]:
    print(f"  [{msg.role}] {msg.content}")

生成摘要: 历史对话摘要：用户: 第 1 轮提问 | 助手: 第 1 轮回答 | 用户: 第 2 轮提问 | 助手: 第 2 轮回答 | 用户: 第 3 轮提问 | 助手: 第 3 轮回答
合并后上下文条数: 3
  [system] 历史对话摘要：用户: 第 1 轮提问 | 助手: 第 1 轮回答 | 用户: 第 2 轮提问 | 助手: 第 2 轮回答 | 用户: 第 3 轮提问 | 助手: 第 3 轮回答
  [user] 第 4 轮提问
  [assistant] 第 4 轮回答


## 4. 验证 `SQLiteChatMessageHistory` 多租户隔离与 `RunnableWithMessageHistory` 闭环
模拟财务部（`tenant-fin`）与研发部（`tenant-rd`）并发调用同一问答链，验证数据库落盘与零串台隔离。

In [4]:
def enterprise_qa_chain(payload: dict) -> str:
    hist_preview = [m.content for m in payload["history"]]
    return f"[{payload['session_id']}] 收到『{payload['input']}』| 注入历史={hist_preview}"

chain = RunnableWithMessageHistory(
    runnable=enterprise_qa_chain,
    get_session_history=lambda sid: SQLiteChatMessageHistory(session_id=sid, db_path=db_path),
    memory_strategy=WindowMemory(k=1),
)

print(chain.invoke("差旅住宿标准是多少？", config={"configurable": {"session_id": "tenant-fin"}}))
print(chain.invoke("发票抬头怎么开？", config={"configurable": {"session_id": "tenant-fin"}}))
print(chain.invoke("GitLab 分支规范是什么？", config={"configurable": {"session_id": "tenant-rd"}}))

sessions = SQLiteChatMessageHistory.list_sessions(db_path)
print("\nSQLite 中隔离的会话列表:", sessions)
for sid in sessions:
    msgs = SQLiteChatMessageHistory(sid, db_path).messages
    print(f"  会话 {sid} 落盘消息数: {len(msgs)}")

tmp_dir.cleanup()

[tenant-fin] 收到『差旅住宿标准是多少？』| 注入历史=[]
[tenant-fin] 收到『发票抬头怎么开？』| 注入历史=['差旅住宿标准是多少？', '[tenant-fin] 收到『差旅住宿标准是多少？』| 注入历史=[]']
[tenant-rd] 收到『GitLab 分支规范是什么？』| 注入历史=[]

SQLite 中隔离的会话列表: ['tenant-fin', 'tenant-rd']
  会话 tenant-fin 落盘消息数: 4
  会话 tenant-rd 落盘消息数: 2
